In [1]:

import sys
import subprocess
import json
import hashlib
import shutil
import textwrap
from pathlib import Path
from datetime import datetime, timezone
from xml.sax.saxutils import escape

try:
    import pandas as pd
    import numpy as np

    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "pandas", "numpy", "reportlab", "pyarrow"
    ])
    import pandas as pd
    import numpy as np

    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )

# ============================================================
# 1) PROJECT ROOT DISCOVERY
# ============================================================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
PREPARED_ROOT = PROJECT_ROOT / "data" / "prepared"
TRANSFORMED_ROOT = PROJECT_ROOT / "data" / "transformed"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
SILVER_ROOT = PROJECT_ROOT / "data" / "silver"
FEATURES_ROOT = PROJECT_ROOT / "data" / "features"
REMEDIATED_ROOT = PROJECT_ROOT / "data" / "remediated"

REPORTS_DIR = PROJECT_ROOT / "reports"
VALIDATION_DIR = REPORTS_DIR / "validation"
LOGS_DIR = PROJECT_ROOT / "logs"
SRC_DIR = PROJECT_ROOT / "src"

VERSIONING_DIR = PROJECT_ROOT / "data_versioning"
VERSIONING_DIR.mkdir(parents=True, exist_ok=True)

REGISTRY_DIR = VERSIONING_DIR / "registry"
REGISTRY_DIR.mkdir(parents=True, exist_ok=True)

DOCS_DIR = VERSIONING_DIR / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc)
RUN_ID = RUN_TS.strftime("%Y%m%dT%H%M%SZ")

OUTPUT_FILE_NAME = "08 Data Versioning and Lineage- DM4ML-Group51.pdf"
OUTPUT_PATH = PROJECT_ROOT / OUTPUT_FILE_NAME

LINEAGE_REGISTRY_CSV = REGISTRY_DIR / "dataset_lineage_registry.csv"
LINEAGE_REGISTRY_JSON = REGISTRY_DIR / "dataset_lineage_registry.json"
VERSIONING_WORKFLOW_MD = DOCS_DIR / "versioning_workflow.md"
VERSIONING_CONFIG_JSON = REGISTRY_DIR / "versioning_config.json"
VERSIONING_LOG = LOGS_DIR / f"data_versioning_log_{RUN_ID}.jsonl"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"VERSIONING_DIR: {VERSIONING_DIR}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")

# ============================================================
# 2) GENERIC HELPERS
# ============================================================
def safe_str(x):
    try:
        return str(x)
    except Exception:
        return ""

def rel_path(path):
    try:
        return safe_str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        try:
            return safe_str(Path(path))
        except Exception:
            return safe_str(path)

def log_event(stage, status, message, extra=None):
    record = {
        "event_ts": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "status": status,
        "message": message,
        "extra": extra or {},
    }
    VERSIONING_LOG.parent.mkdir(parents=True, exist_ok=True)
    with open(VERSIONING_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def collect_files(base_dir, patterns):
    results = []
    if not Path(base_dir).exists():
        return results
    for pattern in patterns:
        results.extend(Path(base_dir).rglob(pattern))
    return sorted(set(p for p in results if p.is_file()))

def latest_file(base_dir, patterns):
    files = collect_files(base_dir, patterns)
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

def file_info(path):
    if not path or not Path(path).exists():
        return None
    path = Path(path)
    st = path.stat()
    return {
        "name": path.name,
        "relative_path": rel_path(path),
        "size_kb": round(st.st_size / 1024, 2),
        "modified": datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "suffix": path.suffix.lower(),
    }

def read_text_preview(path, max_lines=80, max_chars=12000):
    if not path or not Path(path).exists():
        return "File not found."
    try:
        text = Path(path).read_text(encoding="utf-8", errors="ignore")
        lines = text.splitlines()[:max_lines]
        return "\n".join(lines)[:max_chars]
    except Exception as e:
        return f"Could not preview file: {e}"

def read_json_preview(path, max_chars=12000):
    if not path or not Path(path).exists():
        return "JSON file not found."
    try:
        with open(path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        return json.dumps(payload, indent=2, ensure_ascii=False)[:max_chars]
    except Exception as e:
        return f"Could not read JSON preview: {e}"

def read_notebook_preview(path, max_code_cells=4, max_chars=9000):
    if not path or not Path(path).exists():
        return "Notebook not found."
    try:
        with open(path, "r", encoding="utf-8") as f:
            nb = json.load(f)
        blocks = []
        code_idx = 0
        for cell in nb.get("cells", []):
            if cell.get("cell_type") != "code":
                continue
            code_idx += 1
            src = cell.get("source", [])
            src = "".join(src) if isinstance(src, list) else str(src)
            src = src.strip()
            if src:
                blocks.append(f"# Code cell {code_idx}\n{src}")
            if len("\n\n".join(blocks)) >= max_chars or code_idx >= max_code_cells:
                break
        out = "\n\n".join(blocks).strip()
        return out[:max_chars] if out else "No code cells found."
    except Exception as e:
        return f"Could not parse notebook: {e}"

def read_code_preview(path, max_lines=140, max_chars=9000):
    if not path or not Path(path).exists():
        return "File not found."
    if Path(path).suffix.lower() == ".ipynb":
        return read_notebook_preview(path, max_code_cells=4, max_chars=max_chars)
    return read_text_preview(path, max_lines=max_lines, max_chars=max_chars)

def build_tree_text(base_path, max_depth=5, max_items=250):
    base_path = Path(base_path)
    if not base_path.exists():
        return f"{base_path.name}/ (not found)"
    lines = [f"{base_path.name}/"]
    count = 0

    def walk(path, prefix="", depth=0):
        nonlocal count
        if depth >= max_depth or count >= max_items:
            return
        items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        for idx, item in enumerate(items):
            if count >= max_items:
                break
            connector = "└── " if idx == len(items) - 1 else "├── "
            lines.append(prefix + connector + item.name + ("/" if item.is_dir() else ""))
            count += 1
            if item.is_dir():
                extension = "    " if idx == len(items) - 1 else "│   "
                walk(item, prefix + extension, depth + 1)

    walk(base_path)
    if count >= max_items:
        lines.append("... output truncated ...")
    return "\n".join(lines)

def wrap_block_text(text, width=95):
    wrapped = []
    for line in str(text).splitlines():
        if not line.strip():
            wrapped.append("")
            continue
        pieces = textwrap.wrap(
            line,
            width=width,
            break_long_words=True,
            break_on_hyphens=True,
            replace_whitespace=False,
            drop_whitespace=False,
        )
        wrapped.extend(pieces if pieces else [""])
    return "\n".join(wrapped)

def make_hashable_value(v):
    if isinstance(v, np.ndarray):
        return tuple(make_hashable_value(x) for x in v.tolist())
    if isinstance(v, list):
        return tuple(make_hashable_value(x) for x in v)
    if isinstance(v, tuple):
        return tuple(make_hashable_value(x) for x in v)
    if isinstance(v, set):
        return tuple(sorted(make_hashable_value(x) for x in v))
    if isinstance(v, dict):
        return json.dumps(v, sort_keys=True, ensure_ascii=False, default=str)
    try:
        hash(v)
        return v
    except TypeError:
        return str(v)

def make_display_value(v):
    if isinstance(v, np.ndarray):
        return json.dumps(v.tolist(), ensure_ascii=False)
    if isinstance(v, (list, tuple, set)):
        try:
            return json.dumps(list(v), ensure_ascii=False)
        except Exception:
            return str(v)
    if isinstance(v, dict):
        try:
            return json.dumps(v, ensure_ascii=False, sort_keys=True, default=str)
        except Exception:
            return str(v)
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    return safe_str(v)

def safe_duplicate_count(df):
    if df is None or df.empty:
        return 0
    tmp = df.copy()
    for col in tmp.columns:
        tmp[col] = tmp[col].map(make_hashable_value)
    return int(tmp.duplicated().sum())

def wrap_path_for_pdf(value, max_chunk=32):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    separators = {"\\", "/", "_", "-", "="}
    parts = []
    token = ""

    for ch in text:
        token += ch
        if ch in separators:
            parts.append(token)
            token = ""
    if token:
        parts.append(token)

    lines = []
    current = ""

    for part in parts:
        if len(current) + len(part) <= max_chunk:
            current += part
        else:
            if current:
                lines.append(current)
            if len(part) <= max_chunk:
                current = part
            else:
                subparts = textwrap.wrap(
                    part,
                    width=max_chunk,
                    break_long_words=True,
                    break_on_hyphens=True,
                )
                if subparts:
                    lines.extend(subparts[:-1])
                    current = subparts[-1]
                else:
                    current = part

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def wrap_general_text_for_pdf(value, max_len=36):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    words = text.split()
    lines = []
    current = ""

    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_len:
            current = candidate
        else:
            if current:
                lines.append(current)
            if len(word) > max_len:
                chunks = textwrap.wrap(
                    word,
                    width=max_len,
                    break_long_words=True,
                    break_on_hyphens=True,
                )
                if chunks:
                    lines.extend(chunks[:-1])
                    current = chunks[-1]
                else:
                    current = word
            else:
                current = word

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def sha256_file(path, chunk_size=1024 * 1024):
    if not path or not Path(path).exists():
        return None
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def parse_partition_value(path_text, key):
    marker = f"{key}="
    if marker not in path_text:
        return None
    after = path_text.split(marker, 1)[1]
    value = after.split("\\", 1)[0].split("/", 1)[0]
    return value

def infer_source_system(path_text):
    txt = path_text.lower()
    if "retailrocket" in txt:
        return "retailrocket"
    if "dummyjson" in txt:
        return "dummyjson"
    if "validation" in txt:
        return "validation"
    return "unknown"

def infer_stage(path):
    txt = rel_path(path).lower()
    if "data\\raw" in txt or "data/raw" in txt:
        return "raw"
    if "data\\bronze" in txt or "data/bronze" in txt:
        return "bronze"
    if "data\\prepared" in txt or "data/prepared" in txt:
        return "prepared"
    if "data\\processed" in txt or "data/processed" in txt:
        return "processed"
    if "data\\silver" in txt or "data/silver" in txt:
        return "silver"
    if "data\\transformed" in txt or "data/transformed" in txt:
        return "transformed"
    if "data\\features" in txt or "data/features" in txt:
        return "features"
    if "data\\remediated" in txt or "data/remediated" in txt:
        return "remediated"
    if "reports\\validation" in txt or "reports/validation" in txt:
        return "validation_report"
    return "other"

def detect_format(path):
    suffix = Path(path).suffix.lower()
    return suffix.replace(".", "") if suffix else "unknown"

# ============================================================
# 3) PROJECT DISCOVERY
# ============================================================
def discover_dataset_files():
    bases = [
        RAW_ROOT, BRONZE_ROOT, PREPARED_ROOT, PROCESSED_ROOT,
        SILVER_ROOT, TRANSFORMED_ROOT, FEATURES_ROOT, REMEDIATED_ROOT
    ]
    patterns = ["**/*.csv", "**/*.parquet", "**/*.json", "**/*.jsonl"]
    files = []
    for base in bases:
        if base.exists():
            for pattern in patterns:
                files.extend(base.rglob(pattern))
    return sorted(set(p for p in files if p.is_file()))

def discover_versioning_assets():
    patterns = [
        ".dvc/config",
        ".dvcignore",
        ".gitattributes",
        "*.dvc",
        "*.gitkeep",
        "*version*.py", "*version*.ipynb", "*version*.sql", "*version*.md",
        "*lineage*.py", "*lineage*.ipynb", "*lineage*.sql", "*lineage*.md",
        "*dvc*.py", "*dvc*.ipynb", "*dvc*.yaml", "*dvc*.yml", "*dvc*.md",
        "*gitlfs*.py", "*gitlfs*.md", "*git_lfs*.md",
    ]
    matches = []
    for base in [PROJECT_ROOT, SRC_DIR]:
        if base.exists():
            for pattern in patterns:
                matches.extend(base.rglob(pattern))
    cleaned = []
    for p in sorted(set(matches)):
        p_str = safe_str(p).lower()
        if ".ipynb_checkpoints" in p_str:
            continue
        if "/venv/" in p_str or "\\venv\\" in p_str or "/.venv/" in p_str or "\\.venv\\" in p_str:
            continue
        if "/site-packages/" in p_str or "\\site-packages\\" in p_str:
            continue
        cleaned.append(p)
    return cleaned

def read_git_head():
    head_file = PROJECT_ROOT / ".git" / "HEAD"
    if not head_file.exists():
        return None
    try:
        head_text = head_file.read_text(encoding="utf-8", errors="ignore").strip()
        if head_text.startswith("ref:"):
            ref_path = head_text.split("ref:", 1)[1].strip()
            ref_file = PROJECT_ROOT / ".git" / Path(ref_path)
            if ref_file.exists():
                commit = ref_file.read_text(encoding="utf-8", errors="ignore").strip()
                return {"head_ref": ref_path, "commit_hash": commit}
        return {"head_ref": "detached", "commit_hash": head_text}
    except Exception:
        return None

def parse_dvc_file(path):
    if not path or not Path(path).exists():
        return {}
    try:
        text = Path(path).read_text(encoding="utf-8", errors="ignore")
        lines = [line.rstrip() for line in text.splitlines()]
        info = {
            "dvc_file": rel_path(path),
            "outs_path": None,
            "md5": None,
            "size": None,
        }
        for line in lines:
            s = line.strip()
            if s.startswith("path:") and info["outs_path"] is None:
                info["outs_path"] = s.split("path:", 1)[1].strip()
            elif s.startswith("md5:") and info["md5"] is None:
                info["md5"] = s.split("md5:", 1)[1].strip()
            elif s.startswith("size:") and info["size"] is None:
                info["size"] = s.split("size:", 1)[1].strip()
        return info
    except Exception:
        return {}

# ============================================================
# 4) BUILD DATASET LINEAGE REGISTRY
# ============================================================
def build_lineage_registry(dataset_files):
    rows = []
    for path in dataset_files:
        rp = rel_path(path)
        info = file_info(path)
        stage = infer_stage(path)
        source_system = infer_source_system(rp)
        load_date = parse_partition_value(rp, "load_date")
        load_hour = parse_partition_value(rp, "load_hour")

        transformation = []
        if stage == "raw":
            transformation.append("initial_ingestion")
        if stage == "bronze":
            transformation.append("format_standardization")
        if stage in {"prepared", "processed", "silver"}:
            transformation.append("cleaning_preprocessing")
        if stage in {"transformed", "features"}:
            transformation.append("feature_transformation")
        if stage == "remediated":
            transformation.append("automated_remediation")
        if stage == "validation_report":
            transformation.append("validation_reporting")
        if not transformation:
            transformation.append("unknown")

        version_label = "unversioned"
        dvc_pointer = None
        if Path(str(path) + ".dvc").exists():
            version_label = "dvc_tracked"
            dvc_pointer = rel_path(str(path) + ".dvc")

        rows.append({
            "dataset_name": path.name,
            "relative_path": rp,
            "stage": stage,
            "source_system": source_system,
            "file_format": detect_format(path),
            "size_kb": info["size_kb"] if info else None,
            "last_modified": info["modified"] if info else None,
            "load_date": load_date,
            "load_hour": load_hour,
            "ingestion_date_inferred": load_date,
            "applied_transformations": " -> ".join(transformation),
            "versioning_tool": "dvc" if dvc_pointer else "custom_registry",
            "version_label": version_label,
            "dvc_pointer_file": dvc_pointer,
            "sha256": sha256_file(path),
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["stage", "source_system", "dataset_name", "relative_path"]).reset_index(drop=True)
    return df

def build_lineage_summary(lineage_df):
    if lineage_df is None or lineage_df.empty:
        return pd.DataFrame([{"metric": "datasets_discovered", "value": 0}])

    summary = [
        {"metric": "datasets_discovered", "value": len(lineage_df)},
        {"metric": "raw_datasets", "value": int((lineage_df["stage"] == "raw").sum())},
        {"metric": "bronze_datasets", "value": int((lineage_df["stage"] == "bronze").sum())},
        {"metric": "prepared_processed_datasets", "value": int(lineage_df["stage"].isin(["prepared", "processed", "silver"]).sum())},
        {"metric": "transformed_feature_datasets", "value": int(lineage_df["stage"].isin(["transformed", "features"]).sum())},
        {"metric": "remediated_datasets", "value": int((lineage_df["stage"] == "remediated").sum())},
        {"metric": "dvc_tracked_datasets", "value": int((lineage_df["version_label"] == "dvc_tracked").sum())},
        {"metric": "unique_source_systems", "value": int(lineage_df["source_system"].nunique())},
    ]
    return pd.DataFrame(summary)

def create_versioning_workflow_doc(lineage_df, versioning_assets, git_info, dvc_rows):
    workflow_lines = []
    workflow_lines.append("# Data Versioning and Lineage Workflow")
    workflow_lines.append("")
    workflow_lines.append("## Objective")
    workflow_lines.append("Version raw and transformed datasets, track lineage metadata, and document how datasets move through the RecoMart pipeline.")
    workflow_lines.append("")
    workflow_lines.append("## Recommended Workflow")
    workflow_lines.append("1. Ingest source data into partitioned raw folders under data/raw using source and load timestamp conventions.")
    workflow_lines.append("2. Store standardized outputs in bronze and downstream prepared/transformed/features locations.")
    workflow_lines.append("3. Version datasets with DVC where available, or maintain this custom registry with hashes and metadata when DVC pointers are absent.")
    workflow_lines.append("4. Record metadata including dataset name, path, source system, ingestion date, stage, transformations, and file hash.")
    workflow_lines.append("5. Preserve validation reports, fix logs, and revalidation summaries in reports/validation for auditability.")
    workflow_lines.append("6. Regenerate lineage_registry artifacts after each ingestion or transformation run.")
    workflow_lines.append("")
    workflow_lines.append("## Project Observations")
    workflow_lines.append(f"- Datasets discovered: {0 if lineage_df is None else len(lineage_df)}")
    workflow_lines.append(f"- Versioning assets discovered: {len(versioning_assets)}")
    workflow_lines.append(f"- DVC pointer files discovered: {len(dvc_rows)}")
    if git_info:
        workflow_lines.append(f"- Git HEAD ref: {git_info.get('head_ref')}")
        workflow_lines.append(f"- Git commit hash: {git_info.get('commit_hash')}")
    workflow_lines.append("")
    workflow_lines.append("## Metadata Fields Captured")
    workflow_lines.append("- dataset_name")
    workflow_lines.append("- relative_path")
    workflow_lines.append("- stage")
    workflow_lines.append("- source_system")
    workflow_lines.append("- file_format")
    workflow_lines.append("- size_kb")
    workflow_lines.append("- last_modified")
    workflow_lines.append("- load_date / load_hour inferred from partitioned paths when present")
    workflow_lines.append("- applied_transformations")
    workflow_lines.append("- versioning_tool")
    workflow_lines.append("- version_label")
    workflow_lines.append("- dvc_pointer_file")
    workflow_lines.append("- sha256")
    workflow_lines.append("")
    workflow_lines.append("## Usage Notes")
    workflow_lines.append("- Prefer DVC for large dataset versioning and reproducible pulls/checkouts.")
    workflow_lines.append("- Use Git for code, metadata, and pointer files rather than large raw binaries.")
    workflow_lines.append("- If Git LFS is present, restrict it to large binary artifacts that do not fit normal Git workflows.")
    workflow_lines.append("- Keep lineage registry generation as part of the pipeline so metadata stays current.")
    text = "\n".join(workflow_lines)
    VERSIONING_WORKFLOW_MD.write_text(text, encoding="utf-8")
    return text

def build_versioning_config(lineage_df, git_info, dvc_rows):
    cfg = {
        "project": "RecoMart Data Versioning and Lineage",
        "created_ts": RUN_TS.isoformat(),
        "registry_csv": rel_path(LINEAGE_REGISTRY_CSV),
        "registry_json": rel_path(LINEAGE_REGISTRY_JSON),
        "workflow_doc": rel_path(VERSIONING_WORKFLOW_MD),
        "git": git_info or {},
        "dvc_pointer_files": dvc_rows,
        "dataset_counts": {} if lineage_df is None or lineage_df.empty else lineage_df["stage"].value_counts().to_dict(),
    }
    VERSIONING_CONFIG_JSON.write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")
    return cfg

# ============================================================
# 5) PDF STYLES AND TABLE HELPERS
# ============================================================
styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    name="CustomTitle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    fontSize=16,
    leading=20,
    spaceAfter=14,
)

meta_style = ParagraphStyle(
    name="MetaStyle",
    parent=styles["Normal"],
    alignment=TA_LEFT,
    fontSize=10.2,
    leading=13,
    spaceAfter=5,
)

heading_style = ParagraphStyle(
    name="HeadingStyle",
    parent=styles["Heading2"],
    alignment=TA_LEFT,
    fontSize=12,
    leading=15,
    spaceAfter=8,
)

sub_heading_style = ParagraphStyle(
    name="SubHeadingStyle",
    parent=styles["Heading3"],
    alignment=TA_LEFT,
    fontSize=10.2,
    leading=12.5,
    spaceAfter=6,
)

body_style = ParagraphStyle(
    name="BodyStyle",
    parent=styles["BodyText"],
    alignment=TA_JUSTIFY,
    fontSize=10.0,
    leading=14,
    spaceAfter=8,
)

bullet_style = ParagraphStyle(
    name="BulletStyle",
    parent=styles["BodyText"],
    alignment=TA_LEFT,
    fontSize=10.0,
    leading=14,
    leftIndent=14,
    firstLineIndent=-8,
    spaceAfter=4,
)

code_style = ParagraphStyle(
    name="CodeStyle",
    parent=styles["Code"],
    fontName="Courier",
    fontSize=7.0,
    leading=8.4,
)

table_header_style = ParagraphStyle(
    name="TableHeaderStyle",
    parent=styles["BodyText"],
    fontName="Helvetica-Bold",
    fontSize=8.1,
    leading=9.4,
    alignment=TA_LEFT,
)

table_cell_style = ParagraphStyle(
    name="TableCellStyle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=7.0,
    leading=8.5,
    alignment=TA_LEFT,
)

def to_para(value, style, kind="general"):
    if kind == "path":
        return Paragraph(wrap_path_for_pdf(value), style)
    return Paragraph(wrap_general_text_for_pdf(value), style)

def make_wrapped_table(data, col_widths=None, header_bg="#D9EAD3", path_cols=None, file_cols=None):
    path_cols = path_cols or []
    file_cols = file_cols or []
    converted = []

    for r, row in enumerate(data):
        row_cells = []
        for c, cell in enumerate(row):
            style = table_header_style if r == 0 else table_cell_style
            if r == 0:
                row_cells.append(Paragraph(escape(str(cell)), style))
            else:
                if c in path_cols or c in file_cols:
                    row_cells.append(to_para(cell, style, kind="path"))
                else:
                    row_cells.append(to_para(cell, style, kind="general"))
        converted.append(row_cells)

    table = Table(converted, colWidths=col_widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(header_bg)),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    return table

def df_to_wrapped_table(df, max_rows=20, col_widths=None):
    if df is None or df.empty:
        data = [["No data available"]]
    else:
        preview = df.head(max_rows).copy()
        for col in preview.columns:
            preview[col] = preview[col].map(make_display_value)
        data = [list(preview.columns)] + preview.astype(str).values.tolist()
    return make_wrapped_table(data, col_widths=col_widths)

# ============================================================
# 6) BUILD VERSIONING / LINEAGE ARTIFACTS
# ============================================================
log_event("lineage", "started", "Building data versioning and lineage artifacts")

dataset_files = discover_dataset_files()
versioning_assets = discover_versioning_assets()
git_info = read_git_head()

dvc_pointer_files = collect_files(PROJECT_ROOT, ["**/*.dvc"])
dvc_rows = []
for p in dvc_pointer_files:
    parsed = parse_dvc_file(p)
    if parsed:
        dvc_rows.append(parsed)

lineage_df = build_lineage_registry(dataset_files)
lineage_summary_df = build_lineage_summary(lineage_df)

LINEAGE_REGISTRY_CSV.write_text("", encoding="utf-8")
if lineage_df is not None and not lineage_df.empty:
    lineage_df.to_csv(LINEAGE_REGISTRY_CSV, index=False)
    LINEAGE_REGISTRY_JSON.write_text(lineage_df.to_json(orient="records", indent=2, force_ascii=False), encoding="utf-8")
else:
    pd.DataFrame(columns=[
        "dataset_name", "relative_path", "stage", "source_system", "file_format",
        "size_kb", "last_modified", "load_date", "load_hour", "ingestion_date_inferred",
        "applied_transformations", "versioning_tool", "version_label", "dvc_pointer_file", "sha256"
    ]).to_csv(LINEAGE_REGISTRY_CSV, index=False)
    LINEAGE_REGISTRY_JSON.write_text("[]", encoding="utf-8")

workflow_text = create_versioning_workflow_doc(lineage_df, versioning_assets, git_info, dvc_rows)
versioning_config = build_versioning_config(lineage_df, git_info, dvc_rows)

log_event("lineage", "completed", "Lineage registry and workflow documentation created", {
    "registry_csv": rel_path(LINEAGE_REGISTRY_CSV),
    "registry_json": rel_path(LINEAGE_REGISTRY_JSON),
    "workflow_md": rel_path(VERSIONING_WORKFLOW_MD),
})

# ============================================================
# 7) COLLECT PROJECT / LOG EVIDENCE
# ============================================================
latest_validation_txt = latest_file(PROJECT_ROOT, ["**/run_validation.txt"])
latest_validation_json = latest_file(VALIDATION_DIR, ["**/data_quality_report_*.json"])
latest_validation_pdf = latest_file(VALIDATION_DIR, ["**/data_quality_report_*.pdf"])
latest_fix_log = latest_file(VALIDATION_DIR, ["**/fix_log_*.csv"])
latest_reval_summary = latest_file(VALIDATION_DIR, ["**/validation_summary_revalidated_*.csv"])
latest_reval_issues = latest_file(VALIDATION_DIR, ["**/validation_issues_revalidated_*.csv"])
latest_validation_log = latest_file(LOGS_DIR, ["**/validation_log_*.jsonl"])

project_tree_text = build_tree_text(PROJECT_ROOT, max_depth=3, max_items=220)
raw_tree_text = build_tree_text(RAW_ROOT, max_depth=5, max_items=180)
downstream_tree_text = "\n\n".join([
    build_tree_text(x, max_depth=4, max_items=120)
    for x in [BRONZE_ROOT, PREPARED_ROOT, TRANSFORMED_ROOT, FEATURES_ROOT, REMEDIATED_ROOT]
    if x.exists()
]) if any(x.exists() for x in [BRONZE_ROOT, PREPARED_ROOT, TRANSFORMED_ROOT, FEATURES_ROOT, REMEDIATED_ROOT]) else "No downstream data folders found."

versioning_tree_text = build_tree_text(VERSIONING_DIR, max_depth=5, max_items=160)

validation_preview = read_text_preview(latest_validation_txt, max_lines=80, max_chars=10000)
validation_json_preview = read_json_preview(latest_validation_json, max_chars=10000) if latest_validation_json else "No validation JSON report found."
validation_log_preview = read_text_preview(latest_validation_log, max_lines=80, max_chars=10000) if latest_validation_log else "No validation event log found."
workflow_preview = read_text_preview(VERSIONING_WORKFLOW_MD, max_lines=200, max_chars=12000)
config_preview = read_json_preview(VERSIONING_CONFIG_JSON, max_chars=10000)
lineage_json_preview = read_json_preview(LINEAGE_REGISTRY_JSON, max_chars=10000)

asset_previews = []
for p in versioning_assets[:5]:
    asset_previews.append({
        "name": p.name,
        "relative_path": rel_path(p),
        "preview": read_code_preview(p, max_lines=140, max_chars=9000),
    })

# ============================================================
# 8) TABLE DATA
# ============================================================
team_data = [
    ["Team Member Name", "Team Member ID"],
    ["BANSHIDHAR RATH", "2025AE05346"],
    ["JITENDRA KUMAR TIWARI", "2025AE05518"],
    ["KATBA ANKIT CHIMANBHAI", "2025AE05229"],
    ["NAVEEN SURATHU", "2025AE05492"],
]

artifact_rows = [["Log / Report File", "Relative Path", "Last Modified", "Size"]]
for p in [
    latest_validation_txt,
    latest_validation_json,
    latest_validation_pdf,
    latest_fix_log,
    latest_reval_summary,
    latest_reval_issues,
    latest_validation_log,
    LINEAGE_REGISTRY_CSV,
    LINEAGE_REGISTRY_JSON,
    VERSIONING_WORKFLOW_MD,
    VERSIONING_CONFIG_JSON,
    VERSIONING_LOG,
]:
    if p and Path(p).exists():
        info = file_info(Path(p))
        artifact_rows.append([
            info["name"],
            info["relative_path"],
            info["modified"],
            f"{info['size_kb']} KB",
        ])
if len(artifact_rows) == 1:
    artifact_rows.append(["No artifacts found", "-", "-", "-"])

versioning_asset_rows = [["Versioning / Lineage Asset", "Relative Path", "Last Modified", "Size"]]
for p in versioning_assets[:20]:
    info = file_info(p)
    versioning_asset_rows.append([
        info["name"],
        info["relative_path"],
        info["modified"],
        f"{info['size_kb']} KB",
    ])
if len(versioning_asset_rows) == 1:
    versioning_asset_rows.append(["No versioning or lineage asset found", "-", "-", "-"])

dvc_rows_table = [["DVC Pointer", "Tracked Output Path", "MD5", "Size"]]
for row in dvc_rows[:20]:
    dvc_rows_table.append([
        row.get("dvc_file", ""),
        row.get("outs_path", ""),
        row.get("md5", ""),
        row.get("size", ""),
    ])
if len(dvc_rows_table) == 1:
    dvc_rows_table.append(["No .dvc files found", "-", "-", "-"])

git_summary_df = pd.DataFrame([{
    "head_ref": git_info.get("head_ref") if git_info else "git_not_detected",
    "commit_hash": git_info.get("commit_hash") if git_info else "",
    "dvc_pointer_files": len(dvc_rows),
    "datasets_registered": 0 if lineage_df is None else len(lineage_df),
}])

lineage_preview_df = lineage_df.copy()
if lineage_preview_df is not None and not lineage_preview_df.empty:
    keep_cols = [
        "dataset_name", "stage", "source_system", "relative_path",
        "ingestion_date_inferred", "applied_transformations", "version_label"
    ]
    lineage_preview_df = lineage_preview_df[keep_cols]

team_table = make_wrapped_table(team_data, col_widths=[4.0 * inch, 2.0 * inch])

artifact_table = make_wrapped_table(
    artifact_rows,
    col_widths=[1.60 * inch, 3.30 * inch, 1.00 * inch, 0.60 * inch],
    path_cols=[1],
    file_cols=[0]
)

versioning_asset_table = make_wrapped_table(
    versioning_asset_rows,
    col_widths=[1.70 * inch, 3.20 * inch, 1.00 * inch, 0.60 * inch],
    path_cols=[1],
    file_cols=[0]
)

dvc_table = make_wrapped_table(
    dvc_rows_table,
    col_widths=[1.35 * inch, 2.55 * inch, 1.45 * inch, 0.85 * inch],
    path_cols=[0, 1],
    file_cols=[]
)

lineage_summary_table = df_to_wrapped_table(lineage_summary_df, max_rows=20)
git_summary_table = df_to_wrapped_table(git_summary_df, max_rows=10)
lineage_registry_table = df_to_wrapped_table(lineage_preview_df, max_rows=25)

# ============================================================
# 9) BUILD PDF STORY
# ============================================================
story = []

story.append(Paragraph("08 Data Versioning and Lineage", title_style))
story.append(Paragraph("<b>Course Name:</b> Data Management for Machine Learning", meta_style))
story.append(Paragraph("<b>Assignment Title:</b> End-to-End Data Management Pipeline for a Recommendation System", meta_style))
story.append(Paragraph("<b>Assignment:</b> Group 51 - Data management for Machine Learning Group 51", meta_style))
story.append(Spacer(1, 10))

story.append(Paragraph("<b>Team Members</b>", heading_style))
story.append(team_table)
story.append(Spacer(1, 14))

story.append(Paragraph("1. Objective", heading_style))
story.append(Paragraph(
    "This report documents the data versioning and lineage stage of the recommendation pipeline. It captures repository structure, discovered dataset versions, lineage metadata, and workflow documentation for reproducibility and auditability.",
    body_style
))

story.append(Paragraph("2. Objective Coverage", heading_style))
story.append(Paragraph("• Detect DVC or Git-based versioning assets where present.", bullet_style))
story.append(Paragraph("• Register raw and downstream datasets with lineage metadata such as source, ingestion date, and transformations.", bullet_style))
story.append(Paragraph("• Generate repository structure views showing dataset locations and version-related artifacts.", bullet_style))
story.append(Paragraph("• Produce documentation of the versioning workflow for submission.", bullet_style))
story.append(Spacer(1, 10))

story.append(Paragraph("3. Supporting Project and Log Artifacts", heading_style))
story.append(Paragraph(
    "The following discovered files support this section, including validation outputs, lineage registry artifacts, and workflow documentation. Long paths are wrapped using real line breaks only at safe separators to avoid cut-off text in the PDF.",
    body_style
))
story.append(artifact_table)
story.append(Spacer(1, 12))

story.append(Paragraph("4. Versioning and Lineage Assets in Project", heading_style))
story.append(versioning_asset_table)
story.append(Spacer(1, 12))

story.append(Paragraph("5. Git / Versioning Summary", heading_style))
story.append(git_summary_table)
story.append(Spacer(1, 12))

story.append(Paragraph("6. DVC Pointer Summary", heading_style))
story.append(dvc_table)
story.append(Spacer(1, 12))

story.append(Paragraph("7. Dataset Lineage Summary", heading_style))
story.append(lineage_summary_table)
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("8. Dataset Lineage Registry Preview", heading_style))
story.append(Paragraph(
    "The registry below documents dataset name, stage, source, inferred ingestion date, applied transformations, and version label.",
    body_style
))
story.append(lineage_registry_table)
story.append(Spacer(1, 12))

story.append(Paragraph("9. Versioning Workflow Summary", heading_style))
workflow_notes = [
    "Raw datasets are discovered from partitioned source folders under data/raw.",
    "Bronze, prepared, transformed, features, and remediated folders are registered as downstream lineage stages.",
    "load_date and load_hour are inferred from partitioned folder names when present.",
    "A SHA256 hash is recorded for each discovered dataset to support traceability.",
    "If a matching .dvc pointer exists, the dataset is labeled as DVC-tracked.",
    "If no DVC pointer exists, the notebook still builds a custom lineage registry for documentation and audit use.",
    "Validation outputs and fix logs are included as supporting evidence in reports/validation and logs.",
]
for note in workflow_notes:
    story.append(Paragraph(f"• {escape(note)}", bullet_style))
story.append(Spacer(1, 12))

story.append(Paragraph("10. Repository Structure Showing Dataset Versions", heading_style))
story.append(Paragraph("<b>Project Root Structure</b>", meta_style))
story.append(Preformatted(wrap_block_text(project_tree_text, width=92), code_style))
story.append(Spacer(1, 8))
story.append(Paragraph("<b>Raw Data Structure</b>", meta_style))
story.append(Preformatted(wrap_block_text(raw_tree_text, width=92), code_style))
story.append(Spacer(1, 8))
story.append(Paragraph("<b>Downstream Data Structure</b>", meta_style))
story.append(Preformatted(wrap_block_text(downstream_tree_text, width=92), code_style))
story.append(Spacer(1, 8))
story.append(Paragraph("<b>Versioning Artifact Structure</b>", meta_style))
story.append(Preformatted(wrap_block_text(versioning_tree_text, width=92), code_style))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("11. Versioning Workflow Documentation", heading_style))
story.append(Preformatted(wrap_block_text(workflow_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(Paragraph("12. Versioning Configuration Preview", heading_style))
story.append(Preformatted(wrap_block_text(config_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(Paragraph("13. Lineage Registry JSON Preview", heading_style))
story.append(Preformatted(wrap_block_text(lineage_json_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("14. Versioning / Lineage Code or Asset Preview", heading_style))
if asset_previews:
    for item in asset_previews:
        story.append(Paragraph(f"Asset: {escape(item['name'])}", sub_heading_style))
        story.append(Paragraph(f"<b>Path:</b> {escape(item['relative_path'])}", meta_style))
        story.append(Preformatted(wrap_block_text(item["preview"], width=95), code_style))
        story.append(Spacer(1, 10))
else:
    story.append(Paragraph("No existing versioning or lineage notebook/script preview is available.", body_style))
story.append(Spacer(1, 8))

story.append(Paragraph("15. Validation / Pipeline Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("16. Data Quality Report Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_json_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("17. Validation Event Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_log_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("18. Data Versioning Event Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(read_text_preview(VERSIONING_LOG, max_lines=80, max_chars=10000), width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("19. Conclusion", heading_style))
story.append(Paragraph(
    "This PDF consolidates the Data Versioning and Lineage deliverables by documenting dataset locations, inferred versions, lineage metadata, repository structure, and a reproducible versioning workflow suitable for submission.",
    body_style
))

# ============================================================
# 10) BUILD PDF
# ============================================================
def build_pdf(path):
    doc = SimpleDocTemplate(
        str(path),
        pagesize=A4,
        rightMargin=0.50 * inch,
        leftMargin=0.50 * inch,
        topMargin=0.55 * inch,
        bottomMargin=0.55 * inch,
    )
    doc.build(story)

try:
    build_pdf(OUTPUT_PATH)
    print(f"\nPDF created successfully: {OUTPUT_PATH}")
except PermissionError:
    alt_path = PROJECT_ROOT / f"08 Data Versioning and Lineage- DM4ML-Group51-{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    build_pdf(alt_path)
    print("\nOriginal PDF is likely open or locked.")
    print(f"Saved alternate file instead: {alt_path}")


PROJECT_ROOT: C:\Users\barath\recomart-pipeline
VERSIONING_DIR: C:\Users\barath\recomart-pipeline\data_versioning
OUTPUT_PATH: C:\Users\barath\recomart-pipeline\08 Data Versioning and Lineage- DM4ML-Group51.pdf

PDF created successfully: C:\Users\barath\recomart-pipeline\08 Data Versioning and Lineage- DM4ML-Group51.pdf
